In [1]:
import sys, os
try:
    _d = os.path.dirname(os.path.abspath(__vsc_ipynb_file__))
except NameError:
    _d = os.getcwd()
while _d != os.path.dirname(_d):
    if os.path.exists(os.path.join(_d, '_config.yml')): break
    _d = os.path.dirname(_d)
if _d not in sys.path: sys.path.insert(0, _d)
from ai_widget_loader import load_ai_widget
load_ai_widget()

# יצירת טבלה וטעינת נתונים

נתחיל מטבלה קטנה של תכונות פיזיקליות של תרכובות: שם, נוסחה, משפחה כימית, מסה מולרית, נקודת התכה ונקודת רתיחה. אלה נתונים טבלאיים טבעיים: לכל שורה יש תרכובת אחת, וכל עמודה מתארת תכונה אחת.


```{note}
אפשר להוריד את קובץ הנתונים לעבודה עצמאית: {download}`compound_phase_points.csv <files/compound_phase_points.csv>`.
```


## יצירת `DataFrame` ממילון של רשימות

כאשר הנתונים קטנים מאוד, אפשר ליצור טבלה ישירות בקוד. במילון של רשימות, כל מפתח במילון הופך לשם עמודה, וכל רשימה הופכת לערכים באותה עמודה.


In [2]:
import pandas as pd

small_data = {
    "compound": ["water", "ethanol", "hexane"],
    "formula": ["H2O", "C2H6O", "C6H14"],
    "boiling_point_C": [100.0, 78.4, 68.7],
}

small_df = pd.DataFrame(small_data)
display(small_df)


,compound,formula,boiling_point_C
0,water,H2O,100.0
1,ethanol,C2H6O,78.4
2,hexane,C6H14,68.7


שימו לב: המספרים שמופיעים בצד שמאל הם **שמות השורות**. אם לא קבענו שמות בעצמנו, Pandas נותנת לשורות שמות מספריים: `0`, `1`, `2`, וכן הלאה. אלה אינם חלק מהנתונים הכימיים עצמם.


## טעינת טבלה מקובץ CSV

בפועל, לרוב לא נקליד טבלאות ידנית. נקבל קובץ נתונים ונקרא אותו לפייתון. קובץ CSV הוא קובץ טקסט שבו כל שורה היא רשומה, והערכים מופרדים בפסיקים.


In [3]:
data_path = "files/compound_phase_points.csv"
df = pd.read_csv(data_path)
display(df)


,compound,formula,family,molar_mass_g_mol,melting_point_C,boiling_point_C,polarity
0,water,H2O,inorganic,18.02,0.0,100.0,polar
1,methanol,CH4O,alcohol,32.04,-97.6,64.7,polar
2,ethanol,C2H6O,alcohol,46.07,-114.1,78.4,polar
3,1-propanol,C3H8O,alcohol,60.10,-126.0,97.2,polar
4,acetone,C3H6O,ketone,58.08,-94.7,56.1,polar
5,diethyl ether,C4H10O,ether,74.12,-116.3,34.6,weakly polar
6,hexane,C6H14,alkane,86.18,-95.0,68.7,nonpolar
7,heptane,C7H16,alkane,100.20,-90.6,98.4,nonpolar
8,benzene,C6H6,aromatic,78.11,5.5,80.1,nonpolar
9,toluene,C7H8,aromatic,92.14,-95.0,110.6,nonpolar


המשתנה `df` הוא `DataFrame`. כל שורה מייצגת תרכובת אחת, וכל עמודה מייצגת תכונה של התרכובת.


## מבט ראשון על הטבלה

לפני שמבצעים חישוב, כדאי לבדוק מה יש בטבלה. `head` מציגה את השורות הראשונות:


In [4]:
display(df.head())


,compound,formula,family,molar_mass_g_mol,melting_point_C,boiling_point_C,polarity
0,water,H2O,inorganic,18.02,0.0,100.0,polar
1,methanol,CH4O,alcohol,32.04,-97.6,64.7,polar
2,ethanol,C2H6O,alcohol,46.07,-114.1,78.4,polar
3,1-propanol,C3H8O,alcohol,60.10,-126.0,97.2,polar
4,acetone,C3H6O,ketone,58.08,-94.7,56.1,polar


המאפיין `shape` אומר כמה שורות וכמה עמודות יש בטבלה. התוצאה היא זוג: `(number_of_rows, number_of_columns)`.


In [5]:
print(df.shape)


(16, 7)


המאפיין `dtypes` מציג את טיפוס הנתונים בכל עמודה. לדוגמה, עמודות מספריות יהיו לרוב `float64`, ועמודות טקסט כמו שם התרכובת יהיו `object`.


In [6]:
print(df.dtypes)


compound             object
formula              object
family               object
molar_mass_g_mol    float64
melting_point_C     float64
boiling_point_C     float64
polarity             object
dtype: object


## שמות עמודות ושמות שורות

שמות העמודות נשמרים ב־`df.columns`, ושמות השורות נשמרים ב־`df.index`:


In [7]:
print(df.columns)
print(df.index)


Index(['compound', 'formula', 'family', 'molar_mass_g_mol', 'melting_point_C',
       'boiling_point_C', 'polarity'],
      dtype='object')
RangeIndex(start=0, stop=16, step=1)


במחברות הבאות נשתמש בשמות האלה כדי לבחור חלקים מתוך הטבלה. הרעיון המרכזי הוא פשוט: במקום לזכור שעמודה מסוימת נמצאת במקום מספר 5, אפשר לבקש את העמודה לפי השם שלה, למשל `"boiling_point_C"`.
